In [ ]:
# F5-TTS continuation on many Persian speakers.
#
# What this run tests: the Persian checkpoint reads Persian better than the
# corpus it is scored against (2.9% CER against a 7.8% recogniser floor) and
# transfers identity poorly (0.520 held-out against a 0.689 human ceiling).
# That is what training on many hours of few speakers produces. If so,
# continuing on 2739 speakers should recover speaker generalisation.
#
# Decision rule, fixed in advance: held-out similarity must move by more than
# 0.0205 (the encoder's own noise) while CER stays at or below 7.8%.
!pip -q install f5-tts


In [ ]:
import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["WANDB_DISABLED"] = "true"

import f5_tts
import torch

# f5_tts resolves its data and checkpoint directories relative to the package
# as "<pkg>/../..", so every path below has to be anchored the same way or the
# trainer silently looks somewhere else.
#
# __path__[0], not __file__: f5_tts is a namespace package, so __file__ is None
# and Path(None) raises. str(files("f5_tts")) is no good either - it returns the
# MultiplexedPath repr rather than a path.
BASE = Path(list(f5_tts.__path__)[0]).resolve().parent.parent

DATASET_NAME = "persian_multispeaker"
TOKENIZER = "custom"
DATA_DIR = BASE / "data" / f"{DATASET_NAME}_{TOKENIZER}"
CKPT_DIR = BASE / "ckpts" / DATASET_NAME
WORKING = Path("/kaggle/working")

print("f5_tts base :", BASE)
print("data dir    :", DATA_DIR)
print("ckpt dir    :", CKPT_DIR)
print("torch       :", torch.__version__)
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("gpus        :", torch.cuda.device_count(), names)

# Two GPUs is part of the recipe, not a preference. batch_size_per_gpu is 12000
# frames because that is what produced the checkpoint being continued, and the
# effective batch is that times the number of processes - so a single card
# halves it and changes the optimisation dynamics against weights already
# adapted to them. Falling back to one card would be a silent quality cut, so
# stop here instead and go fix the accelerator.
#
# The card matters too: a generic "Gpu" request yields one P100 (sm_60), which
# recent torch builds no longer support.
assert torch.cuda.device_count() == 2, (
    f"Need 2 GPUs, got {torch.cuda.device_count()}: {names}. "
    "Set machine_shape to the T4 x2 accelerator and push again - training on "
    "one card would halve the effective batch."
)


In [ ]:
# Kaggle may present the attached dataset as plain directories or as archives
# depending on how it was uploaded, so find it rather than assume a layout.
import tarfile
import zipfile


def locate() -> Path:
    roots = sorted(Path("/kaggle/input").glob("*"))
    for root in roots:
        for meta in root.rglob("metadata.csv"):
            if (meta.parent / "wavs").is_dir():
                return meta.parent
    # Nothing unpacked: look for an archive and extract it once.
    target = Path("/kaggle/working/dataset")
    for root in roots:
        archives = (
            list(root.rglob("*.tar.gz")) + list(root.rglob("*.tar")) + list(root.rglob("*.zip"))
        )
        for archive in archives:
            print("extracting", archive)
            target.mkdir(parents=True, exist_ok=True)
            if archive.suffix == ".zip":
                with zipfile.ZipFile(archive) as bundle:
                    bundle.extractall(target)
            else:
                with tarfile.open(archive) as bundle:
                    bundle.extractall(target)
            for meta in target.rglob("metadata.csv"):
                if (meta.parent / "wavs").is_dir():
                    return meta.parent
    raise SystemExit(f"No dataset found under /kaggle/input: {[p.name for p in roots]}")

SRC = locate()
info_path = SRC / "dataset_info.json"
info = json.loads(info_path.read_text()) if info_path.is_file() else {}
wav_count = len(list((SRC / "wavs").glob("*.wav")))
print("dataset     :", SRC)
print("wavs        :", wav_count)
print("info        :", {k: info.get(k) for k in ("clips", "speakers", "hours", "download_failures")})
assert wav_count > 1000, f"only {wav_count} wavs - the upload is incomplete"


In [ ]:
# f5_tts's prepare script is strict about its input: it demands a header line
# reading exactly "audio_file|text" and rejects any path that is not absolute.
# Our manifest is "<id>|<transcript>", so translate rather than hand it over.
rows = []
missing = 0
with (SRC / "metadata.csv").open(encoding="utf-8") as handle:
    for row in csv.reader(handle, delimiter="|"):
        if len(row) < 2:
            continue
        wav = SRC / "wavs" / f"{row[0]}.wav"
        if not wav.is_file():
            missing += 1
            continue
        rows.append((str(wav.resolve()), " ".join(row[1].split())))

TRAIN_CSV = WORKING / "train.csv"
with TRAIN_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle, delimiter="|")
    writer.writerow(["audio_file", "text"])
    writer.writerows(rows)

print(f"{len(rows)} rows written, {missing} manifest rows had no wav")
assert rows, "nothing to train on"


In [ ]:
from huggingface_hub import hf_hub_download

raw_ckpt = hf_hub_download("Lumos675/F5_TTS_Persian", "model_last.pt")
VOCAB = Path(hf_hub_download("Lumos675/F5_TTS_Persian", "vocab.txt"))

# THE TRAP. The published checkpoint is a full training state: it carries
# "update": 420000 alongside the optimizer and scheduler. f5_tts's
# Trainer.load_checkpoint branches on the presence of that key - if it is there
# it restores the optimizer and scheduler and returns 420000 as the starting
# update. train() then computes skipped_epoch = 420000 // steps_per_epoch,
# which lands far past our epoch count, so `range(skipped_epoch, epochs)` is
# empty and the run finishes without a single gradient step. It reports no
# error. Stripping everything except the EMA weights sends load_checkpoint down
# the other branch, which loads the weights and returns update 0.
state = torch.load(raw_ckpt, map_location="cpu", weights_only=True)
print("published keys:", list(state.keys()))
print("published update:", state.get("update"))

PRETRAIN = WORKING / "pretrained_persian_ema.pt"
torch.save({"ema_model_state_dict": state["ema_model_state_dict"]}, PRETRAIN)
del state

check = torch.load(PRETRAIN, map_location="cpu", weights_only=True)
assert "update" not in check and "step" not in check, "training state survived the strip"
assert "optimizer_state_dict" not in check, "optimizer state survived the strip"
n_ema = len(check["ema_model_state_dict"])
del check
print(f"stripped -> {PRETRAIN.name}, {n_ema} EMA tensors, no training state")

# The vocabulary is not optional: it is a character tokenizer extended with the
# Persian alphabet, and the base F5 vocabulary contains no Persian at all.
vocab_size = sum(1 for _ in VOCAB.open(encoding="utf-8"))
print("vocab size  :", vocab_size, "(checkpoint text embedding is this + 1)")


In [ ]:
# --pretrain here means "not a finetune", which is the opposite of what it
# sounds like: it stops the script copying the Emilia Chinese/English vocab in
# as our vocab.txt. The file it writes instead is ignored - the trainer is
# pointed straight at the Persian vocabulary below.
DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, "-m", "f5_tts.train.datasets.prepare_csv_wavs",
    str(TRAIN_CSV), str(DATA_DIR), "--pretrain",
]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

shutil.copy2(VOCAB, DATA_DIR / "vocab.txt")
duration = json.loads((DATA_DIR / "duration.json").read_text())["duration"]
print("\nprepared    :", DATA_DIR)
print("clips       :", len(duration))
print("hours       :", round(sum(duration) / 3600, 2))


In [ ]:
# The trainer writes to <pkg>/../../ckpts/<dataset>, which on Kaggle is inside
# the Python installation and is thrown away when the session ends. Only
# /kaggle/working survives, so point one at the other before training starts
# rather than trying to rescue the files afterwards.
PERSIST = WORKING / "ckpts"
PERSIST.mkdir(parents=True, exist_ok=True)
CKPT_DIR.parent.mkdir(parents=True, exist_ok=True)
if CKPT_DIR.is_symlink():
    CKPT_DIR.unlink()
elif CKPT_DIR.exists():
    shutil.rmtree(CKPT_DIR)
CKPT_DIR.symlink_to(PERSIST, target_is_directory=True)

shutil.copy2(PRETRAIN, PERSIST / PRETRAIN.name)
print("checkpoints ->", CKPT_DIR, "->", PERSIST)
print("contents    :", [p.name for p in PERSIST.iterdir()])


In [ ]:
import shlex

from f5_tts.train.finetune_cli import parse_args

# Every hyperparameter below is what produced the checkpoint being continued
# (its setting.json): lr 7.5e-5, 12000 frames per GPU, max_samples 64,
# grad accumulation 1, grad norm 1, fp16. Changing them mid-run would change
# the optimisation dynamics against weights already adapted to them.
#
# --finetune is load-bearing and easy to miss: without it finetune_cli never
# assigns ckpt_path, --pretrain is ignored outright, and the run trains from
# random initialisation while looking perfectly healthy.
#
# 80 rather than 40 because the corpus came out at 12.5 h, not the 25 h the
# plan assumed - Common Voice publishes 9985 of the 34134 clips it lists. Half
# the audio at the same epoch count is half the gradient updates, and speaker
# generalisation is the slow half of what this run is trying to move. Doubling
# the epochs restores the training budget the recipe was written for; it does
# not add anything the plan did not already call for.
EPOCHS = 80
WARMUP = 2000

# No --logger. Its choices are [None, "wandb", "tensorboard"] - the Python None
# object, which no command-line string can ever match, so `--logger none` is
# rejected by argparse with exit code 2 and the flag is unusable from a shell.
# Omitting it lands on the same default. WANDB_DISABLED is set above.
train_args = [
    "--exp_name", "F5TTS_v1_Base",
    "--dataset_name", DATASET_NAME,
    "--finetune",
    "--pretrain", str(PERSIST / PRETRAIN.name),
    "--tokenizer", "custom",
    "--tokenizer_path", str(VOCAB),
    "--learning_rate", "7.5e-5",
    "--batch_size_per_gpu", "12000",
    "--batch_size_type", "frame",
    "--max_samples", "64",
    "--grad_accumulation_steps", "1",
    "--max_grad_norm", "1",
    "--epochs", str(EPOCHS),
    "--num_warmup_updates", str(WARMUP),
    "--save_per_updates", "5000",
    "--last_per_updates", "2000",
    "--keep_last_n_checkpoints", "2",
]

# Hand the arguments to the real parser before spending a GPU on them. An
# unusable flag costs a whole queued run to discover otherwise, and argparse
# reports it as a bare exit code 2 from inside accelerate, several tracebacks
# deep. This is the same check, in milliseconds.
saved_argv = sys.argv
sys.argv = ["finetune_cli.py", *train_args]
try:
    parse_args()
    print("arguments accepted by finetune_cli's own parser")
except SystemExit as exit_code:
    raise RuntimeError(f"finetune_cli rejected these arguments (exit {exit_code.code})") from None
finally:
    sys.argv = saved_argv

command = "accelerate launch --mixed_precision=fp16 -m f5_tts.train.finetune_cli " + " ".join(
    shlex.quote(a) for a in train_args
)
print("\n" + command, flush=True)


In [ ]:
outcome = subprocess.run(command, shell=True, text=True)
print("exit code:", outcome.returncode)
assert outcome.returncode == 0, "training failed - read the log above"


In [ ]:
# What to bring home. model_last.pt is the resume point; the numbered ones are
# the intermediate saves. Anything left here is what the session persists.
saved = sorted(PERSIST.glob("*.pt"))
for path in saved:
    print(f"{path.name:40s} {path.stat().st_size / 1e9:6.2f} GB")
if not any(p.name.startswith("model_") for p in saved):
    print("\nNo model_*.pt was written - training did not reach a save point.")
else:
    print("\nFetch the output with: kaggle kernels output <user>/persian-f5-multispeaker-train -p ./remote")
